# Niceleyici Laboratuvarı
## Herkes için ayrı bir tanık mı, herkes için aynı tanık mı?
**MAT507 · Matematiksel Düşünme ve Muhakeme · Niceleyiciler bölümü**

**Süre:** 35–45 dakika. **Ön koşul:** “her”, “en az bir”, “değil” kavramları; Python bilmek zorunlu değil.

Colab'da **Dosya → Drive'a bir kopya kaydet** seçin. Önce aşağıdaki tahmin alanlarını doldurun; sonra hücrelerin solundaki ▶ düğmesini kullanın. Etkileşim için “Çalışma zamanı → Tümünü çalıştır” seçeneği de kullanılabilir.

Bu atölyede ∀/∃ sırasını karşılaştıracak, tanık ve karşı örnek bulacak, olumsuzlama kurallarını sınayacak ve sonlu deney ile sonsuz evrende ispatı ayıracaksınız.

**Çalışma ilkesi:** Tahmin et → dene → çürütmeye çalış → gerekçelendir → yeni soru üret.

Cevaplarınız bu not defterinde kalır; atölye yanıtları bir sunucuya göndermez. Paylaşılan bir bağlantının kendi kopyanızı görünür kılabileceğini unutmayın.

## 1 · Tahmin et
Dört öğrenci ve dört soru var. ✓, öğrencinin soruyu çözdüğünü gösteriyor.

| Öğrenci / soru | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| Ali | ✓ | – | – | – |
| Banu | – | ✓ | – | – |
| Can | – | – | ✓ | – |
| Deniz | – | – | – | ✓ |

P(x,y): “x öğrencisi y sorusunu çözdü.”

- **A:** ∀x ∃y P(x,y) — Her öğrenci en az bir soru çözdü.
- **B:** ∃y ∀x P(x,y) — Bütün öğrencilerin çözdüğü bir soru var.

**Çalıştırmadan önce yazın:** A doğru mu? B doğru mu? B'nin doğru olması için tabloya en az kaç ✓ eklemelisiniz? Neden?

**İlk tahminim:** …

**Gerekçem:** …

In [ ]:
from IPython.display import display, HTML
from html import escape
from itertools import product

OGRENCILER = ["Ali", "Banu", "Can", "Deniz"]
SORULAR = [1, 2, 3, 4]

def cozumle(tablo):
    """Satırlar x, sütunlar y; boş olmayan dikdörtgen bir 0/1 tablosu."""
    if not tablo or not tablo[0] or any(len(s) != len(tablo[0]) for s in tablo):
        raise ValueError("Boş olmayan, dikdörtgen bir tablo kullanın.")
    if any(v not in (0, 1) for s in tablo for v in s):
        raise ValueError("Hücreler 0 veya 1 olmalı.")
    her_satir = all(any(s) for s in tablo)
    ortak_sutunlar = [j for j in range(len(tablo[0])) if all(s[j] for s in tablo)]
    bos_satirlar = [i for i, s in enumerate(tablo) if not any(s)]
    return {
        "A": her_satir,
        "B": bool(ortak_sutunlar),
        "ortak_sutunlar": ortak_sutunlar,
        "bos_satirlar": bos_satirlar,
        "taniklar": [[j for j, v in enumerate(s) if v] for s in tablo],
    }

def goster(tablo):
    r = cozumle(tablo)
    h = '<table style="border-collapse:collapse"><caption>P(x,y): öğrenci soruyu çözdü</caption>'
    h += '<tr><th>Öğrenci / soru</th>' + ''.join(f'<th style="padding:10px">{s}</th>' for s in SORULAR) + '</tr>'
    for ad, satir in zip(OGRENCILER, tablo):
        h += '<tr><th>' + escape(ad) + '</th>'
        for v in satir:
            renk = '#dcfce7' if v else '#f1f5f9'
            h += f'<td style="padding:12px;text-align:center;background:{renk}">{"✓" if v else "–"}</td>'
        h += '</tr>'
    display(HTML(h + '</table>'))
    print("A · ∀x ∃y:", r['A'], " | B · ∃y ∀x:", r['B'])
    if r['A']:
        print("Her öğrenci için seçilebilecek tanık sorular:")
        for ad, js in zip(OGRENCILER, r['taniklar']):
            print(" ", ad, "→", [SORULAR[j] for j in js])
    else:
        print("A'yı çürüten öğrenciler:", [OGRENCILER[i] for i in r['bos_satirlar']])
    if r['B']:
        print("B için ortak tanık sorular:", [SORULAR[j] for j in r['ortak_sutunlar']])
    else:
        print("B neden yanlış? Her soruda bir karşı örnek öğrenci:")
        for j, soru in enumerate(SORULAR):
            i = next(i for i, s in enumerate(tablo) if not s[j])
            print(" ", soru, "→", OGRENCILER[i])

tablo = [[int(i == j) for j in range(4)] for i in range(4)]
goster(tablo)

## 2 · Tabloyu değiştir
Aşağıdaki kutulara dokunarak çözülmüş soruları değiştirin. Sonuçlar ve tanıklar yenilenir.

1. A doğru, B yanlış bir tablo kurun.
2. Yalnızca **üç kutu ekleyerek** başlangıç tablosunda B'yi doğru yapın.
3. A'yı yanlış yapın. Hangi öğrenci karşı örnek oldu?
4. **B doğru, A yanlış** bir tablo kurmaya çalışın. Başaramamak bir ispat mı? Genel gerekçeyi yazın.

Widget görünmezse sonraki kod hücresindeki 0/1 tablosunu değiştirerek aynı deneyleri yapabilirsiniz.

In [ ]:
import ipywidgets as w

kutular = [[w.Checkbox(value=bool(tablo[i][j]), description=str(j+1), indent=False,
                       layout=w.Layout(width="65px")) for j in range(4)] for i in range(4)]
cikti = w.Output()

def yenile(change=None):
    with cikti:
        cikti.clear_output(wait=True)
        goster([[int(k.value) for k in satir] for satir in kutular])

for satir in kutular:
    for k in satir:
        k.observe(yenile, names="value")

satirlar = [w.HBox([w.Label(ad, layout=w.Layout(width="65px"))] + satir)
            for ad, satir in zip(OGRENCILER, kutular)]
display(w.VBox(satirlar + [cikti]))
yenile()

In [ ]:
# Widget kullanmadan deney: 1 = çözdü, 0 = çözmedi.
benim_tablom = [
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
]
goster(benim_tablom)

**Düzelttiğim düşünce:** …

**B doğruysa A neden zorunlu olarak doğrudur?** “Ortak tanık” kavramını kullanarak yazın: …

**Üç ek kutunun yeterli ve gerekli olduğuna dair gerekçem:** …

## 3 · Olumsuzlama dedektifi
“Her öğrenci en az bir soru çözdü” ifadesinin olumsuzu hangisi?

1. Hiçbir öğrenci hiçbir soru çözmedi.
2. En az bir öğrenci hiçbir soru çözmedi.
3. Her öğrencinin çözemediği bir soru var.

**Seçimim ve gerekçem:** …

Şimdi sonlu tablolarda şu eşdeğerliği kontrol edeceğiz:

¬∀x ∃y P(x,y) ⇔ ∃x ∀y ¬P(x,y).

`all` = bütün elemanlar doğru, `any` = en az bir eleman doğru. Aşağıdaki sonuç bütün boyutlar için bir ispat değildir; yalnızca 2×3 boyutundaki bütün tabloların eksiksiz kontrolüdür.

In [ ]:
kontrol_edilen = 0
for hucreler in product([False, True], repeat=6):
    t = [list(hucreler[:3]), list(hucreler[3:])]
    sol = not all(any(satir) for satir in t)
    sag = any(all(not v for v in satir) for satir in t)
    assert sol == sag
    assert not cozumle(t)["B"] or cozumle(t)["A"]
    kontrol_edilen += 1
print(f"{kontrol_edilen} tablonun tamamında olumsuzlama eşdeğerliği ve B ⇒ A doğrulandı.")

## 4 · Sonlu dünya tuzağı
Doğal sayıları bu atölyede ℕ = {0,1,2,…} olarak alıyoruz.

C: ∀x∈ℕ ∃y∈ℕ (y>x).

Kodda yalnızca D={0,…,N} kümesini kullanırsak C'nin sonlu karşılığı doğru olur mu? Sonucu tahmin edin; sonra N'yi değiştirin. **x ve y'nin evrenleri aynı anda sınırlandırılıyor.**

In [ ]:
N = 5  # 0 ile 100 arasında başka bir sayı deneyin.
assert isinstance(N, int) and 0 <= N <= 100
D = list(range(N + 1))
basarisiz_x = [x for x in D if not any(y > x for y in D)]
print("Sonlu evren:", D)
print("∀x∈D ∃y∈D (y>x):", not basarisiz_x)
print("Bu evrende tanığı bulunamayan x:", basarisiz_x)

**Deney ile ispatı ayır:**

- Sonlu D'de neden hep bir sorun çıktı? …
- ℕ'de her x için hangi y'yi seçebiliriz? …
- Bu seçimin doğal sayı olduğunu ve y>x sağladığını açıklayın: …
- ∃y∈ℕ ∀x∈ℕ (y>x) neden yanlıştır? Savunucu bir y seçtiğinde Şüpheci hangi x'i seçer? …

N'yi büyütmek sonlu bir evreni sonsuz yapmaz. ℕ için gerekçe, her x'e uygulanabilen bir tanık kuralı gerektirir.

## 5 · Kendi iddianı üret ve çıkış bileti
1. Yeni bir P(x,y) ilişkisi tanımlayın; x ve y'nin evrenlerini açıkça yazın.
2. ∀x∃y P(x,y) ve ∃y∀x P(x,y) cümlelerini günlük dille yazın.
3. Bir tanık veya karşı örnek gösterin. Sonucunuz sonlu bir kontrol mü, genel bir gerekçe mi?

**İlişkim ve evrenlerim:** …

**İki cümlem:** …

**Tanığım / karşı örneğim ve gerekçem:** …

**Çıkış bileti:** “İlk başta … düşünüyordum. … deneyinden sonra … fark ettim. Bunu yalnızca örnekle değil, … gerekçesiyle savunabilirim.”

Kendi kontrolünüz: Evreni belirttim mi? Tanığın neye bağlı olduğunu söyledim mi? Gözlem ile ispatı ayırdım mı?

[Kitaba ve bölüm sayfasına dön](https://github.com/ibrahimguney/matematik-dusunme/tree/main/atolyeler/niceleyiciler)